In [1]:
import os, random, torch, numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training Environment Initialized on: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

Training Environment Initialized on: cpu (CPU)


In [2]:
WARMUP_EPOCHS = 3
FINETUNE_EPOCHS = 12
BATCH_SIZE = 32
BASE_LR = 1e-3
BACKBONE_LR = 1e-5
LABEL_SMOOTHING = 0.1

print(f"Hyperparameters: Batch={BATCH_SIZE} | LR_Head={BASE_LR} | LR_Backbone={BACKBONE_LR} | LabelSmooth={LABEL_SMOOTHING}")

Hyperparameters: Batch=32 | LR_Head=0.001 | LR_Backbone=1e-05 | LabelSmooth=0.1


In [3]:
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

# Step A: Export FP32 ONNX
dummy_input = torch.randn(1, 3, 224, 224, device="cpu")
torch.onnx.export(
    model.to("cpu"),
    dummy_input,
    "efficientnet_b0_fp32.onnx",
    opset_version=14,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)

# Step B: Run Dynamic INT8 Quantization
quantize_dynamic(
    model_input="efficientnet_b0_fp32.onnx",
    model_output="efficientnet_b0_quantized.onnx",
    weight_type=QuantType.QInt8,
    per_channel=True
)

fp32_sz = os.path.getsize("efficientnet_b0_fp32.onnx") / (1024 * 1024)
int8_sz = os.path.getsize("efficientnet_b0_quantized.onnx") / (1024 * 1024)
print(f"FP32 Size: {fp32_sz:.2f} MB ---> Quantized INT8 Size: {int8_sz:.2f} MB (Compression: {(1 - int8_sz/fp32_sz)*100:.1f}%)")

NameError: name 'model' is not defined